# Membrane proteins and variant-effect predictors: talk figures

One message per figure, written for slides. Every headline is computed from the numbers below it, so it always says what this dataset shows. PNG and PDF files go to `figures_talk_<name>/`, tables to `figures_talk_<name>/tables/`. Under each figure, *slide notes* give the numbers to quote.

Sections: 1 structure · 2 clinical performance · 3 where predictors fail · 4 per-protein failures · 5 how predictors relate · 6 calibration · 7 VUS · 8 ThermoMPNN's worst proteins (clinical) · 9 ThermoMPNN's least-agreeing proteins · 10 overlap of the two lists.

In [ ]:
import os, sys, json, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import Markdown, display
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore", category=RuntimeWarning)
from mvsa import analyses as A, stats as S, talk as T, thermo as Th
from mvsa.data import load, available_predictors, dataset_label
from mvsa.groups import MEMBRANE, MAIN_PREDICTORS, DIRECTION

DATA_DIR = Path(os.environ.get("MVSA_DATA", "../data_human"))
FIG_DIR = Path(os.environ.get("MVSA_FIGURES", "../figures_talk"))
N_BOOT = int(os.environ.get("MVSA_NBOOT", 300))
N_BOTTOM = int(os.environ.get("MVSA_NBOTTOM", 100))   # size of the 'worst proteins' lists
MIN_PER_CLASS = 5                                      # per-protein AUC needs this many P and B variants
GOOD_AUC = 0.85                                        # 'ranked well' threshold used in the ThermoMPNN headline

T.style(FIG_DIR)
D = load(DATA_DIR)
TAXID = D["meta"].get("taxid") or 9606
ORGANISM = dataset_label(D["meta"])
STRUCT_SETS = [x for x in os.environ.get("MVSA_STRUCT_SETS", "human,yeast").split(",") if x]   # datasets on the structure figure
lab_all = D["labelled_variants"]
lab = S.labels(lab_all); lab = lab[lab.protein_class.notna()]
sample = D["variant_sample"]
HAS_LABELS = len(lab) >= 20 and lab.y.nunique() == 2
PRED = [p for p in MAIN_PREDICTORS if p in available_predictors(lab if HAS_LABELS else sample)]
VEP = [p for p in ("alphamissense", "esm1v", "saprot") if p in PRED]
PCORR = pd.read_parquet(DATA_DIR / "protein_correlations.parquet") if (DATA_DIR / "protein_correlations.parquet").exists() else None
TABLE_DIR = FIG_DIR / "tables"; TABLE_DIR.mkdir(parents=True, exist_ok=True)

def note(*lines):
    display(Markdown("**Slide notes**\n\n" + "\n".join(f"- {l}" for l in lines)))

def fmt_q(q):
    return "q < 0.001" if q < 0.001 else f"q = {q:.2g}"

print(f"{ORGANISM}: {len(PRED)} predictors | labels: {HAS_LABELS} | per-protein correlations: {PCORR is not None}")
if PCORR is None:
    print("protein_correlations.parquet missing: run lsf/protcorr.sh first; agreement analyses are skipped")

## 1. Membrane proteins are built differently

In [ ]:
def shape_units_for(d):
    meta = json.loads((d / 'meta.json').read_text())
    pr = pd.read_parquet(d / 'proteins.parquet', columns=['accession', 'taxid', 'class_annotation', 'in_ve_results'])
    pr = pr[pr.in_ve_results.astype(bool)] if pr.in_ve_results.any() else pr[pr.taxid == meta['taxid']]
    su = pd.read_parquet(d / 'shape_units.parquet')
    su = su[su.accession.isin(pr.accession)].copy()
    su['protein_class'] = su.accession.map(pr.set_index('accession').class_annotation)
    return su

orgs = {o: DATA_DIR.parent / f'data_{o}' for o in STRUCT_SETS}
orgs = {o: d for o, d in orgs.items() if (d / 'meta.json').exists()} or {ORGANISM: DATA_DIR}
contr = {o: A.structural_contrasts(shape_units_for(d)) for o, d in orgs.items()}
fig = T.structure_dots(contr)
c0 = next(iter(contr.values())); c0 = c0[c0['class'] == 'SOLUBLE'].set_index('metric')
big = c0.reindex(list(T.STRUCT_SHOW)).cliffs_delta_size_matched.abs().dropna()
n_big = int((big >= 0.5).sum())
T.headline(fig, 'Membrane proteins are built differently' + (' — in ' + ', '.join(contr) if len(contr) > 1 else ''),
           f'{n_big} of {len(big)} structural features differ strongly (|δ| ≥ 0.5) from size-matched soluble proteins')
T.save(fig, '01_structure'); plt.show()
lines = []
for m in big.sort_values(ascending=False).index[:5]:
    vals = ', '.join(f"{o} {c[(c['class']=='SOLUBLE') & (c.metric==m)].cliffs_delta_size_matched.iat[0]:+.2f}" for o, c in contr.items())
    lines.append(f'{T.STRUCT_SHOW[m]}: δ {vals}')
note(*lines, 'δ = Cliff\'s delta within size bins; +1 means every multipass protein is higher.')

## 2. Clinical performance: no overall deficit

In [ ]:
if HAS_LABELS:
    pooled = A.pooled_performance(lab, PRED, n_boot=N_BOOT)
    so = pooled[pooled['class'] == 'SOLUBLE']
    worse = so[(so.mp_minus_delta_high < 0) & (so.mp_minus_q < 0.05)]
    better = so[(so.mp_minus_delta_low > 0) & (so.mp_minus_q < 0.05)]
    fig = T.auc_dumbbell(pooled, PRED)
    title = ('Clinical AUC is no lower in membrane proteins' if not len(worse)
             else f'{len(worse)} of {len(PRED)} predictors are worse on membrane proteins')
    mpl_ = lab[lab.protein_class == MEMBRANE]
    T.headline(fig, title, f'{int(mpl_.y.sum()):,} pathogenic and {int((mpl_.y == 0).sum()):,} benign ClinVar variants in '
               f'{mpl_.accession.nunique():,} multipass proteins, against soluble proteins')
    T.save(fig, '02_clinical_auc'); plt.show()
    best = pooled[pooled['class'] == MEMBRANE].sort_values('auc', ascending=False).iloc[0]
    note(f'Significantly worse on multipass: {len(worse)}; significantly better: {len(better)} (BH q < 0.05).',
         f'Best on multipass: {T.name(best.predictor)} AUC {best.auc:.3f} ({best.ci_low:.3f}–{best.ci_high:.3f}).',
         'Physicochemical features (BLOSUM62, |Δ| volume/charge/hydrophobicity) sit at 0.54–0.65 in every class: the gap is between predictor types, not protein classes.')
else:
    print('No ClinVar labels for', ORGANISM)

## 3. Where predictors fail: inside vs outside the membrane

In [ ]:
if HAS_LABELS:
    LOCP = [p for p in VEP + ['thermompnn'] if p in PRED]
    loc = A.location_performance(lab, LOCP, n_boot=min(N_BOOT, 200))
    fig = T.location_dots(loc, LOCP)
    g = lambda p, c, l: loc[(loc.predictor == p) & (loc['class'] == c) & (loc.location == l)].auc.mean()
    gap_ext = np.nanmean([g(p, MEMBRANE, 'Extramembrane exposed') - g(p, 'SOLUBLE', 'Exposed') for p in VEP] +
                         [g(p, MEMBRANE, 'Extramembrane buried') - g(p, 'SOLUBLE', 'Buried') for p in VEP])
    gap_tm = np.nanmean([g(p, MEMBRANE, 'TM lipid-facing') - g(p, 'SOLUBLE', 'Exposed') for p in VEP])
    if gap_tm > 0.01 and gap_ext < -0.01:
        title = 'Predictors excel in TM helices but lose ground outside the membrane'
    elif gap_ext < -0.01:
        title = 'Membrane proteins are predicted less well outside the membrane'
    else:
        title = 'Prediction quality by location inside membrane proteins'
    T.headline(fig, title, f'Sequence predictors: TM lipid-facing {gap_tm:+.3f}, extramembrane {gap_ext:+.3f} AUC vs soluble residues of the same exposure')
    T.save(fig, '03_location'); plt.show()
    loc.to_csv(TABLE_DIR / 'auc_by_location.tsv', sep='\t', index=False)
    note(*[f"{T.name(p)}: TM lipid-facing {g(p, MEMBRANE, 'TM lipid-facing'):.3f} · extramembrane exposed {g(p, MEMBRANE, 'Extramembrane exposed'):.3f} "
           f"vs soluble exposed {g(p, 'SOLUBLE', 'Exposed'):.3f} · extramembrane buried {g(p, MEMBRANE, 'Extramembrane buried'):.3f} vs soluble buried {g(p, 'SOLUBLE', 'Buried'):.3f}" for p in LOCP],
         'Check the CIs in the figure before quoting a gap: differences of 0.02-0.04 need overlapping intervals checked.')

## 4. Per-protein failures: how often does a predictor get a protein wrong?

In [ ]:
if HAS_LABELS:
    pp, ppc = A.per_protein_performance(D['protein_auc'], PRED, MIN_PER_CLASS)
    fig, fp = T.frac_poor(pp, PRED, 0.7)
    m = fp[(fp['class'] == MEMBRANE) & fp.predictor.isin(VEP + ['thermompnn'])].sort_values('frac', ascending=False).iloc[0]
    so_ = fp[(fp['class'] == 'SOLUBLE') & (fp.predictor == m.predictor)].frac.iat[0]
    T.headline(fig, f'{T.name(m.predictor)} ranks {m.frac:.0%} of membrane proteins poorly',
               f'Per-protein AUC below 0.7 ({so_:.0%} of soluble proteins); proteins with ≥ {MIN_PER_CLASS} pathogenic and benign variants')
    T.save(fig, '04_per_protein_failures'); plt.show()
    note(*[f"{T.name(r.predictor)}: {r.frac:.0%} of {r.n} multipass vs {fp[(fp['class']=='SOLUBLE') & (fp.predictor==r.predictor)].frac.iat[0]:.0%} soluble"
           for r in fp[fp['class'] == MEMBRANE].sort_values('frac', ascending=False).itertuples()])

## 5. How predictors relate to each other

In [ ]:
corr = A.correlation_by_class(sample, PRED)
if MEMBRANE in corr and 'SOLUBLE' in corr:
    fig, diff = T.corr_panels(corr, PRED)
    (a_, b_), d_ = diff.index[0], diff.iloc[0]
    tm = diff[[('thermompnn' in k) for k in diff.index]]
    T.headline(fig, f"{T.name(b_)} and {T.name(a_)} agree {'more' if d_ > 0 else 'less'} in membrane proteins (Δρ = {d_:+.2f})",
               'Spearman ρ between predictors over a 1% sample of all possible substitutions; scores oriented so higher = more damaging')
    T.save(fig, '05_predictor_correlations'); plt.show()
    pd.concat({c: m for c, m in corr.items() if c in (MEMBRANE, 'SOLUBLE')}).to_csv(TABLE_DIR / 'predictor_correlations.tsv', sep='\t')
    note(*[f'{T.name(b)} – {T.name(a)}: Δρ {v:+.2f}' for (a, b), v in diff.head(5).items()],
         *[f'ThermoMPNN – {T.name(b if a == "thermompnn" else a)}: ρ {corr[MEMBRANE].loc[a, b]:.2f} multipass vs {corr["SOLUBLE"].loc[a, b]:.2f} soluble' for (a, b), v in tm.head(4).items()])

## 6. Scores do not transfer: calibration

In [ ]:
if HAS_LABELS:
    calib, _ = A.calibration(lab, PRED)
    fig, w = T.calibration_bars(calib, PRED)
    k = int((w.ratio > 1).sum())
    T.headline(fig, f'A score means something different on membrane proteins',
               f'Calibration error is higher on multipass proteins for {k} of {len(w)} predictors, after adjusting for ClinVar prevalence')
    T.save(fig, '06_calibration'); plt.show()
    note(*[f'{T.name(p)}: {r[MEMBRANE]:.3f} vs {r["SOLUBLE"]:.3f} ({r.ratio:.1f}×)' for p, r in w.sort_values('ratio', ascending=False).iterrows()],
         'No CIs on these yet: quote the consistent direction, not the size.')

## 7. Variants of uncertain significance

In [ ]:
if HAS_LABELS and len(VEP) >= 2:
    per_vus, agree = A.vus_reclassification(lab_all[lab_all.protein_class.notna()], PRED, VEP)
    fig = T.vus_bars(agree, ', '.join(T.name(v) for v in VEP))
    a = agree.set_index('class')
    n_vus = int((lab_all[lab_all.protein_class == MEMBRANE].clinvar_class == 'vus').sum())
    T.headline(fig, f"{a.loc[MEMBRANE, 'all pathogenic']:.0%} of membrane-protein VUS look pathogenic to every predictor",
               f"vs {a.loc['SOLUBLE', 'all pathogenic']:.0%} for soluble proteins; {n_vus:,} VUS in multipass proteins")
    T.save(fig, '07_vus'); plt.show()
    pv = per_vus.pivot_table(index='predictor', columns='class', values='called_pathogenic')
    note(*[f'{T.name(p)} calls {r[MEMBRANE]:.0%} of multipass VUS pathogenic vs {r["SOLUBLE"]:.0%} soluble' for p, r in pv.iterrows() if p in VEP + ['thermompnn']])

## 8. ThermoMPNN's worst membrane proteins (clinical AUC)

Multipass proteins ranked by ThermoMPNN's own ClinVar AUC; the bottom `N_BOTTOM` are compared with the other multipass proteins that have enough labels. Is the protein hard for every predictor, or only for ThermoMPNN? What does ThermoMPNN track in these proteins, and where are their pathogenic variants?

In [ ]:
if HAS_LABELS:
    tt = Th.protein_table(D, lab, PCORR, MIN_PER_CLASS)
    scored = tt.dropna(subset=['auc_thermompnn'])
    n_b = min(N_BOTTOM, len(scored) // 3)   # keep at least twice as many proteins to compare against
    bot = Th.bottom(scored, 'auc_thermompnn', n_b)
    flag = scored.accession.isin(bot.accession)
    print(f'{len(scored)} multipass proteins have a ThermoMPNN AUC; bottom {len(bot)}: AUC {bot.auc_thermompnn.min():.2f}–{bot.auc_thermompnn.max():.2f}')
    cands = [f'auc_{p}' for p in ('esm1v', 'alphamissense', 'saprot') if f'auc_{p}' in scored]
    other = max(cands, key=lambda c: scored[c].notna().sum())   # the sequence predictor with most per-protein AUCs
    fig = T.thermo_scatter(scored, flag, other)
    good = (bot[other] >= GOOD_AUC).mean()
    T.headline(fig, f'{good:.0%} of ThermoMPNN\'s worst membrane proteins are ranked well by {T.name(other[4:])}',
               f'Bottom {len(bot)} of {len(scored)} multipass proteins by ThermoMPNN AUC; well = {T.name(other[4:])} AUC ≥ {GOOD_AUC}')
    T.save(fig, '08a_thermo_vs_vep'); plt.show()
    auc_cols = [f'auc_{p}' for p in PRED if f'auc_{p}' in scored and p != 'thermompnn']
    fig = T.profile(scored, flag, auc_cols, 'Per-protein AUC (median and IQR)', ref=0.5)
    T.headline(fig, 'Is the protein hard, or only hard for ThermoMPNN?', 'Other predictors on the same proteins')
    T.save(fig, '08b_other_predictors'); plt.show()

In [ ]:
if HAS_LABELS and PCORR is not None:
    rho_cols = [c for c in scored if c.startswith('rho_')]
    fig = T.profile(scored, flag, rho_cols, 'Spearman ρ with ThermoMPNN over all substitutions of each protein (median and IQR)')
    dm = {c: scored.loc[flag, c].median() - scored.loc[~flag, c].median() for c in rho_cols}
    c_ = max(dm, key=lambda c: abs(dm[c]))
    T.headline(fig, f"In its worst proteins ThermoMPNN correlates {'more' if dm[c_] > 0 else 'less'} with {T.name(c_[4:])}",
               f'Median ρ {scored.loc[flag, c_].median():.2f} in the bottom {len(bot)} vs {scored.loc[~flag, c_].median():.2f} in other multipass proteins')
    T.save(fig, '08c_thermo_correlations'); plt.show()

In [ ]:
if HAS_LABELS:
    feats = list(Th.FEATURE_LABEL) + [c for c in scored if c.startswith('rho_')]
    feats = [f for f in feats if f not in ('n_P', 'n_B')]
    cmp_clin = Th.contrast(scored, flag, feats)
    assert len(cmp_clin), 'too few proteins to compare'
    fig = T.effects(cmp_clin, Th.FEATURE_LABEL, top=10)
    top = cmp_clin.iloc[0]
    T.headline(fig, f"Worst proteins differ most in {T.flabel(top.feature, Th.FEATURE_LABEL)}",
               'Features of the bottom proteins vs other multipass proteins (Cliff\'s δ; BH-adjusted q)')
    T.save(fig, '08d_what_differs'); plt.show()
    fc = Th.family_counts(scored, flag)
    fig = T.family_bars(fc)
    f0 = fc.iloc[0]
    T.headline(fig, f'{fc.index[0]}s make up {f0.frac_bottom:.0%} of the worst {len(bot)}',
               f'vs {f0.frac_all:.0%} of all multipass proteins with a ThermoMPNN AUC (family from UniProt)')
    T.save(fig, '08e_families'); plt.show()
    cmp_clin.to_csv(TABLE_DIR / 'thermo_bottom_clinical_features.tsv', sep='\t', index=False)
    note(*[f"{T.flabel(r.feature, Th.FEATURE_LABEL)}: {r.median_bottom:.3g} vs {r.median_rest:.3g} (δ {r.cliffs_delta:+.2f}, {fmt_q(r.q)})" for r in cmp_clin.head(6).itertuples()])

In [ ]:
SHOW = ['rank', 'gene', 'protein_name', 'family', 'length', 'n_tm_helical', 'n_P', 'n_B', 'auc_thermompnn', 'auc_esm1v',
        'auc_saprot', 'auc_alphamissense', 'rho_esm1v', 'rho_alphamissense', 'rho_rsa', 'P_in_TM', 'P_exposed', 'thermo_P_minus_B']
if HAS_LABELS:
    out = bot[[c for c in SHOW if c in bot] + [c for c in bot if c not in SHOW]]
    out.to_csv(TABLE_DIR / f'thermo_bottom{len(bot)}_clinical.tsv', sep='\t', index=False)
    print(f'full table: {TABLE_DIR}/thermo_bottom{len(bot)}_clinical.tsv')
    display(out[[c for c in SHOW if c in out]].head(40).round(3).style.hide(axis='index'))

## 9. Where ThermoMPNN disagrees with the sequence predictors

Every protein (no ClinVar needed): ThermoMPNN's *agreement* is the median Spearman ρ between its scores and those of ESM-1v, SaProt and (human) AlphaMissense over all substitutions. The bottom `N_BOTTOM` multipass proteins by agreement are compared with the rest.

In [ ]:
if PCORR is not None:
    ta = Th.protein_table(D, lab if HAS_LABELS else None, PCORR, MIN_PER_CLASS, classes=(MEMBRANE, 'SOLUBLE'))
    fig = T.agreement_hist(ta)
    mm, ms_ = (ta[ta.class_annotation == c].agreement.median() for c in (MEMBRANE, 'SOLUBLE'))
    T.headline(fig, f"ThermoMPNN agrees {'less' if mm < ms_ else 'more'} with sequence predictors on membrane proteins",
               f'Median per-protein ρ {mm:.2f} multipass vs {ms_:.2f} soluble')
    T.save(fig, '09a_agreement'); plt.show()
    tmp = ta[ta.class_annotation == MEMBRANE].dropna(subset=['agreement'])
    bota = Th.bottom(tmp, 'agreement', min(N_BOTTOM, len(tmp) // 3))
    flag_a = tmp.accession.isin(bota.accession)
    part = {f'rho_{p}' for p in Th.PARTNERS}
    feats_a = [f for f in list(Th.FEATURE_LABEL) + [c for c in tmp if c.startswith('rho_') or c.startswith('auc_')]
               if f not in part and f not in ('agreement', 'n_P', 'n_B')]
    cmp_a = Th.contrast(tmp, flag_a, feats_a)
    assert len(cmp_a), 'too few proteins to compare'
    fig = T.effects(cmp_a, Th.FEATURE_LABEL, top=10)
    top = cmp_a.iloc[0]
    T.headline(fig, f"Low-agreement proteins differ most in {T.flabel(top.feature, Th.FEATURE_LABEL)}",
               f'Bottom {len(bota)} of {len(tmp)} multipass proteins by agreement vs the rest (Cliff\'s δ)')
    T.save(fig, '09b_agreement_what_differs'); plt.show()
    fca = Th.family_counts(tmp, flag_a)
    fig = T.family_bars(fca)
    T.headline(fig, f'{fca.index[0]}s make up {fca.iloc[0].frac_bottom:.0%} of the {len(bota)} least-agreeing proteins',
               f'vs {fca.iloc[0].frac_all:.0%} of all multipass proteins')
    T.save(fig, '09c_agreement_families'); plt.show()
    cols = [c for c in ['rank', 'gene', 'protein_name', 'family', 'length', 'n_tm_helical', 'agreement', 'rho_esm1v', 'rho_saprot',
                        'rho_alphamissense', 'rho_rsa', 'rho_depth_mean', 'rho_plddt_model', 'rho_in_tm', 'plddt_mean', 'frac_excluded', 'auc_thermompnn'] if c in bota]
    bota[cols + [c for c in bota if c not in cols]].to_csv(TABLE_DIR / f'thermo_bottom{len(bota)}_agreement.tsv', sep='\t', index=False)
    cmp_a.to_csv(TABLE_DIR / 'thermo_bottom_agreement_features.tsv', sep='\t', index=False)
    note(*[f"{T.flabel(r.feature, Th.FEATURE_LABEL)}: {r.median_bottom:.3g} vs {r.median_rest:.3g} (δ {r.cliffs_delta:+.2f}, {fmt_q(r.q)})" for r in cmp_a.head(6).itertuples()])
    display(bota[cols].head(40).round(3).style.hide(axis='index'))
else:
    print('run lsf/protcorr.sh first')

## 10. Overlap of the two lists

In [ ]:
if HAS_LABELS and PCORR is not None:
    a_set, b_set = set(bot.accession), set(bota.accession)
    both = a_set & b_set
    # expected overlap if unrelated: only proteins eligible for both rankings count
    elig = set(scored.accession) & set(tmp.accession)
    exp = len(a_set & elig) * len(b_set & elig) / max(len(elig), 1)
    from scipy.stats import hypergeom
    p = hypergeom.sf(len(both) - 1, len(elig), len(b_set & elig), len(a_set & elig)) if both else 1.0
    display(Markdown(f'**{len(both)} proteins are in both lists** (≈{exp:.0f} expected by chance among the {len(elig)} proteins '
                     f'eligible for both; hypergeometric p = {p:.2g}).'))
    ov = tt[tt.accession.isin(both)].merge(ta[['accession', 'agreement']].rename(columns={'agreement': 'agree'}), on='accession', how='left')
    ov = ov.sort_values('auc_thermompnn')
    ov.to_csv(TABLE_DIR / 'thermo_bottom_both_lists.tsv', sep='\t', index=False)
    display(ov[[c for c in ['gene', 'protein_name', 'family', 'auc_thermompnn', 'auc_esm1v', 'agreement', 'rho_rsa', 'P_in_TM'] if c in ov]].round(3).style.hide(axis='index'))